In [4]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace

from IPython.display import Markdown, display

load_dotenv(override=True)

True

Define a tidal agent as a tool

In [2]:
tidal_agent = Agent(
    name = "Tidal Agent",
    instructions = """You are an experienced sailor with excellent knowledge of tidal hights and tidal streams.
        Your job is to decide when the best departure times are for a given passage considering only the tidal set and rate.
        Do not consider wind, sea state or other conditions. 
        Your priority when reviewing a passage plan for tide effects is to optimise the speed and comfort of the boat.
        You must raise any safety concerns you identify.
        The vessel moves at 4 knots and is a small sailboat. Tidal streams exceeding 4 knots should be avoided.
        The vessel draws 1.2m. You must verify the depth will exceed 2.2 throughout the whole journey to include a 1m clearence.
        Advise strongly when a plan suggests a route going against the tide.
        Be aware of compromises with other factors such as weather and safety
        In the cases where a compromise over departure time is necessary consider routes which reduce the effect of the tidal stream""",
    model="gpt-4o-mini"
)

Turn the tidal agent into a tool for the passage planner

In [3]:
tools = [
    tidal_agent.as_tool(
        tool_name = "Tidal_Planner",
        tool_description="Decides on the optimum departure times considering only tidal factors on a passage."
    )
]

In [4]:
agent = Agent(
    name="First Mate",
    instructions="You are a helpful first mate with control over a navigational ui through tools. Your responses through the chat must be clear and consise",
    tools=tools,
    model="gpt-4o-mini")

In [5]:
with trace("Kelder Agent Tests"):
    result = await Runner.run(agent, "Hi there, please help me plan a passage in the Solent. I want to travel from Shepards Bush marina in cowes to bembridge habour on the east side of the isle of white uk")

CancelledError: 

In [ ]:
display(Markdown((result.final_output)))

To plan your passage from Shepards Bush Marina in Cowes to Bembridge Harbour, consider the following:

### Key Considerations:
1. **Tidal Heights**: Ensure a minimum depth of 2.2m for your vessel, which requires at least 1m clearance.
2. **Tidal Streams**: Favorable currents are essential. Avoid times when currents exceed 4 knots against your speed.

### Passage Details:
- **Distance**: Approximately 3 nautical miles.
- **Travel Time**: Roughly 45 minutes at 4 knots.

### Recommended Actions:
1. **Departure Time**:
   - Aim to depart close to high water to take advantage of slack or favorable currents.
   - Check local tide tables for Cowes and Bembridge to identify optimal times.
  
2. **Tidal Conditions**:
   - Depart during the last hour of the ebb tide for the best flow.
   - Monitor the current to ensure it isn’t excessively strong against you.

### Safety Checks:
- Verify depth along the route and around Bembridge.
- Have all navigational aids prepared and operational.

If conditions appear unfavorable, consider waiting for the next suitable tidal window to ensure a smooth passage. Always consult local tide tables for updated information. Safe travels!

Streaming tests

In [ ]:
from openai.types.responses import ResponseTextDeltaEvent

result = Runner.run_streamed(agent, input="Please plan a passage from cowes yatch haven to plymouth")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

To assist you with the passage from Cowes Yacht Haven to Plymouth, I'll need to consider tidal factors. Please allow me a moment to analyze the best departure times.For your passage from Cowes Yacht Haven to Plymouth, here are the key considerations:

### Route & Distance
- **Distance:** Approximately 50 nautical miles.

### Vessel Specifications
- **Speed:** 4 knots
- **Draft:** 1.2 meters (ensure a depth of at least 2.2 meters)

### Tidal Considerations
1. **Tidal Heights:** Ensure depths exceed 2.2 meters along the route.
2. **Tidal Streams:** Avoid streams exceeding 4 knots.

### Optimal Departure Time
- **Recommendation:** Depart shortly after high water in Cowes to take advantage of the favorable current. 

### Safety & Route Planning
1. Verify all depths along the route to avoid shallow areas, especially at low tide.
2. Monitor tidal tables continuously for any changes before departure.

### Backup Plan
If conditions are adverse, consider delaying your departure or adjusting you

# Live agent Testing

In [5]:
import os
os.chdir("../..")

In [6]:
from src.kelder_api.routes.inference.agents import get_agent

chatbot_agent = get_agent()

[FunctionTool(name='Tidal_Planner', description='Decides on the optimum departure times considering only tidal factors on a passage.', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'Tidal_Planner_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x1181b3600>, strict_json_schema=True), FunctionTool(name='Passage_Planner', description='Writes and saves a passage plan to the passage plan card on the dashboard', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'Passage_Planner_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x11874c040>, strict_json_schema=True)]


In [16]:
with trace("Kelder Agent Passage planning tests"):
    result = await Runner.run(chatbot_agent, "Hi, please can you plan me a passage between Cowes and Yarmouth habour on the isle of whyte for tomorrow")

display(Markdown((result.final_output)))

/Users/tombramwell/Documents/Kelder App/kelder_api/kelder_api/src/kelder_api/components/passage_plan/tools.py:16: RuntimeWarning: coroutine 'RedisClient.write_set' was never awaited
  redis_client.write_set("PASSAGE_PLAN", passage_plan)


✅ Passage plan prepared and saved to the dashboard.

Summary (brief, safety-first):
- Recommended departure window: tomorrow mid-morning to midday to ride the favorable tidal stream (plan saved with specific times).
- Estimated sail time: ~1.0–1.5 hours at 6.5 kn (depends on tide/traffic).
- Route: leave Cowes harbour, follow the main Cowes–East Cowes channel, pass the Bramble Patch area keeping the recommended transit channel, cross the central Solent keeping an eye for ferry lanes, and approach Yarmouth via the marked Yarmouth fairway/bar. Waypoints and courses are on the saved plan card.
- Hazards flagged: Brambles/Bramble Bank shoals, strong tidal streams in the central Solent, busy ferry traffic, Yarmouth bar/entrance shoaling. Depths and minimum under-keel clearance are included in the plan—check against your actual draft.
- Safety notes: keep a close watch for ferries and large commercial traffic, maintain VHF listening watch, confirm local Notices to Mariners and charted depths before departing, and delay if weather/tide/visibility are poor.
- If your vessel speed or draft differ from the assumptions (6.5 kn, 2.0 m), or you want a specific departure time, tell me now and I’ll update the plan.

Would you like the exact departure time and tide details added to your nav brief?

In [20]:
from src.kelder_api.components.redis_client.redis_client import RedisClient
redis_client = RedisClient()
async with redis_client.get_connection() as redis:
    size = await redis.zcard("sensor:ts:PASSAGE_PLAN")
    print(f"Redis set size: {size}")


Redis set size: 0
